# FediGAN-ViT — experiment notebook (Kaggle · GPU T4 ×2)

Code for: *FediGAN-ViT: A Federated Conditional GAN and Vision Transformer Framework for Data-Local Retinal Disease Classification with Visual Explanations* (Deepthi A. and Sammulal; under review, International Journal of Intelligent Engineering and Systems). `SPEED = "fast"` reproduces the 19 experiments reported in the paper.

Federated ViT with client-side conditional-GAN augmentation for 4-class fundus classification (EDC dataset).
Every table and figure is generated from logged outputs into `/kaggle/working/fedigan_outputs/`.

## How to run
1. **Settings → Accelerator: GPU T4 ×2**, **Internet: On**, add dataset **`gunavenkatdoddi/eye-diseases-classification`**.
2. Check the **Settings** cell below (`SPEED = "fast"` ≈ 2–4 h is the default).
3. Click **Save Version → Save & Run All (Commit)**. It runs on Kaggle's servers — you can close the browser and laptop.
   Check progress under *View versions → Logs*.
4. Download `fedigan_results_paper.zip` from the version's **Output** tab.

**What happens:** the engine is written to a file, then two background workers train experiments in parallel — one per GPU —
while this notebook prints progress every 2 minutes. When both finish, the analysis cells build all tables, figures and Grad-CAM.

**Resume:** if a run stops early (12 h limit or error), open the notebook, **Add Input → Notebook Output Files →** this notebook's
last version, and commit again. Finished experiments are restored and skipped.

**Results table → paper mapping:** T01 dataset · T02 partition · T03 hyper-parameters · T04 main result · T05 per-class ·
T06 ablation/baselines · T07 significance · T08 robustness · T09 cGAN quality · T10 complexity · T11 sensitivity (full plan only).

In [ ]:
!pip install -q timm grad-cam torch-fidelity "torchmetrics[image]"

In [ ]:
# ============================== SETTINGS (the only cell you normally edit) ==============================
import os, time
PROFILE   = "paper"     # "paper" = real study | "smoke" = 1 round per experiment, quick end-to-end check (~30 min)
SPEED     = "fast"      # "fast"     ~2-4 h : ViT-Small, max 15 rounds, 19 experiments (recommended)
                        # "standard" ~6-8 h : ViT-Base,  max 20 rounds, 32 experiments
                        # "full"     ~11+ h : ViT-Base,  max 20 rounds, 56 experiments (needs 2 sessions)
BUDGET_H  = 10.5        # stop STARTING new experiments after this many hours of session time (Kaggle limit: 12 h)

os.environ.update(FEDIGAN_PROFILE=PROFILE, FEDIGAN_SPEED=SPEED, FEDIGAN_BUDGET_H=str(BUDGET_H))
os.environ.setdefault("FEDIGAN_T0", str(time.time()))       # session clock shared with the GPU workers
RUN_TAG = f"{PROFILE}_{SPEED}"                                # results folder: fedigan_outputs/runs_<RUN_TAG>
print(f"PROFILE={PROFILE} | SPEED={SPEED} | budget {BUDGET_H} h")

## 1 · Write the experiment engine
The full training code (identical to what the workers run). Just run this cell — no editing needed.

In [ ]:
%%writefile /kaggle/working/fedigan_core.py
# FediGAN-ViT engine — written by the notebook; do not edit here, edit the notebook settings instead.
# Imported by the notebook (%run) to define everything, and launched as background GPU workers:
#   FEDIGAN_ROLE=prep                          build cache, pre-train all client cGANs, download backbones
#   FEDIGAN_ROLE=shard FEDIGAN_SHARD=i FEDIGAN_NSHARDS=n   run experiments i, i+n, ... on one GPU
import os, matplotlib
if os.environ.get("FEDIGAN_ROLE"):
    matplotlib.use("Agg")
# ============================== CORE IMPORTS ==============================
import os, re, gc, json, time, math, copy, glob, random, hashlib, shutil, warnings
from pathlib import Path
from collections import defaultdict
from multiprocessing import Pool

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
                             cohen_kappa_score, matthews_corrcoef, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score)

warnings.filterwarnings("ignore")
NOTEBOOK_START = float(os.environ.get("FEDIGAN_T0", time.time()))   # session start, shared with GPU workers
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

# ============================== CONFIGURATION ==============================
PROFILE = os.environ.get("FEDIGAN_PROFILE", "paper")   # "smoke" = every code path, 1 seed, 1 round; "paper" = full study

# ---- paths ----
WORK = Path(os.environ.get("FEDIGAN_WORK", "/kaggle/working" if os.path.isdir("/kaggle/working") else "./work"))
OUT = WORK / "fedigan_outputs"
SPEED = os.environ.get("FEDIGAN_SPEED", "fast")   # fast | standard | full  (set in the notebook's Settings cell)
RUN_TAG = f"{PROFILE}_{SPEED}"                     # results of different model sizes are never mixed
RUNS_DIR = OUT / f"runs_{RUN_TAG}"
for d in ["tables", "figures", "cache", "synthetic", "gan", "models", f"runs_{RUN_TAG}"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)

# ---- data ----
CLASS_NAMES = ["cataract", "diabetic_retinopathy", "glaucoma", "normal"]
SHORT = ["Cataract", "DR", "Glaucoma", "Normal"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE, GAN_SIZE = 224, 64
SPLIT_SEED = 42            # fixed test/val split for every run (comparability)
TEST_FRAC, VAL_FRAC = 0.15, 0.15
GROUP_SPLIT = True         # keep both eyes of a patient / near-duplicates in the same split and client

# ---- federated setup ----
NUM_CLIENTS = 3
ALPHA = 0.5                # Dirichlet concentration (non-IID)
MIN_CLIENT_FRAC = 0.10     # resample partition until every client holds >=10% of training data
AGG_WEIGHT = "real"        # FedAvg weights = number of REAL images per client (synthetic does not inflate weight)

# ---- classifier ----
BACKBONE = "vit_small_patch16_224" if SPEED == "fast" else "vit_base_patch16_224"   # ViT-S/16 or ViT-B/16 (IN-21k pretrained)
PRETRAINED = os.environ.get("FEDIGAN_PRETRAINED", "1") == "1"
BATCH = 32
LR = {"vit": 1e-4, "cnn": 3e-4}
WEIGHT_DECAY = 0.05
DROP_RATE = 0.3
ROUNDS, LOCAL_EPOCHS = (15 if SPEED == "fast" else 20), 1
PATIENCE, MIN_ROUNDS = (4 if SPEED == "fast" else 5), 5   # early stopping on validation macro-AUC
MU = 0.01                            # FedProx proximal coefficient
FOCAL_GAMMA, CB_BETA = 2.0, 0.9999   # class-balanced focal loss
GRAD_CLIP = 1.0

# ---- cGAN ----
LATENT, EMBED = 100, 50
GAN_EPOCHS, GAN_BATCH, GAN_LR, GAN_BETA1 = 100, 32, 2e-4, 0.5
SYNTH_CAP = 100            # max synthetic images per class per client (controls real:synthetic mixing ratio λ)
MIN_REAL_FOR_SYNTH = 1     # never synthesise a class the client has no real images of
FEW_SHOT_WARN = 20         # flag classes with fewer real images than this (GAN unreliable)

# ---- robustness / statistics ----
PARTICIPATION = 0.67       # partial-participation run: each client joins a round with this probability
SEEDS_MAIN = [42, 43, 44, 45, 46]   # proposed model
SEEDS_OTHER = [42, 43, 44]          # every other configuration (paired with proposed on these seeds)
SENS_ALPHAS = [0.1, 1.0]
SENS_MUS = [0.001, 0.1]
BOOTSTRAP_B = 1000
TIME_BUDGET_H = 10.8       # stop launching runs after this (Kaggle hard limit is 12 h)
NUM_WORKERS = int(os.environ.get("FEDIGAN_WORKERS", min(4, os.cpu_count() or 2)))
PLAN_MODE = {"fast": "minimal", "standard": "reduced", "full": "full"}.get(SPEED, "reduced")   # 19 / 32 / 56 experiments

if PROFILE == "smoke":
    ROUNDS, MIN_ROUNDS, PATIENCE = 1, 1, 1
    GAN_EPOCHS = 2
    SEEDS_MAIN, SEEDS_OTHER = [42], [42]
    BOOTSTRAP_B = 100
    TIME_BUDGET_H = 2.0
TIME_BUDGET_H = float(os.environ.get("FEDIGAN_BUDGET_H", TIME_BUDGET_H))
if PLAN_MODE in ("reduced", "minimal") and PROFILE != "smoke":
    SEEDS_MAIN = SEEDS_OTHER = [42, 43, 44]

print(f"PROFILE={PROFILE} | SPEED={SPEED} ({BACKBONE}, {PLAN_MODE} plan) | rounds={ROUNDS} | seeds main={SEEDS_MAIN} other={SEEDS_OTHER} | output={OUT}")
_SHARD = os.environ.get("FEDIGAN_SHARD")
_TAG = "" if _SHARD is None else f"_shard{_SHARD}"
TBL = OUT / f"tables{_TAG}"; FIG = OUT / f"figures{_TAG}"
TBL.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

# ============================== TORCH SETUP ==============================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms.v2 as T
from torchvision import tv_tensors
import timm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

print("torch", torch.__version__, "| timm", timm.__version__, "| device", DEVICE, "| GPUs", N_GPU,
      [torch.cuda.get_device_name(i) for i in range(N_GPU)])

def find_data_root():
    env = os.environ.get("FEDIGAN_DATA")
    if env:
        return env
    for c in sorted(glob.glob("/kaggle/input/**/cataract", recursive=True)):
        p = os.path.dirname(c)
        if all(os.path.isdir(os.path.join(p, k)) for k in CLASS_NAMES):
            return p
    raise FileNotFoundError("EDC dataset not found under /kaggle/input — add 'eye-diseases-classification'.")

def crop_to_fundus(bgr, thr=10):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    ys, xs = np.where(g > thr)
    if len(xs) < 100:
        return bgr
    return bgr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

def preprocess_fundus(bgr):
    """Illumination correction -> non-local-means denoising -> CLAHE (LAB L-channel) -> background mask."""
    mask = (cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY) > 10).astype(np.uint8)
    mask = cv2.erode(mask, np.ones((5, 5), np.uint8))
    blur = cv2.GaussianBlur(bgr, (0, 0), bgr.shape[0] / 30)
    ill = cv2.addWeighted(bgr, 4, blur, -4, 128)
    den = cv2.fastNlMeansDenoisingColored(ill, None, 5, 5, 7, 21)
    l, a, b = cv2.split(cv2.cvtColor(den, cv2.COLOR_BGR2LAB))
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    out = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
    return (out * mask[..., None]).astype(np.uint8)

def dhash(bgr, size=8):
    g = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (size + 1, size), interpolation=cv2.INTER_AREA)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return "".join("1" if b else "0" for b in bits)

def _load_one(path):
    cv2.setNumThreads(0)
    with open(path, "rb") as f:
        data = f.read()
    arr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if arr is None:
        return None
    arr = crop_to_fundus(arr)
    raw = cv2.resize(arr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    proc = preprocess_fundus(raw)
    g64 = cv2.resize(proc, (GAN_SIZE, GAN_SIZE), interpolation=cv2.INTER_AREA)
    return (hashlib.md5(data).hexdigest(), dhash(raw),
            np.ascontiguousarray(raw[..., ::-1]), np.ascontiguousarray(proc[..., ::-1]),
            np.ascontiguousarray(g64[..., ::-1]))

DATA_ROOT = find_data_root()
CACHE = OUT / "cache"
cache_files = [CACHE / f for f in ["meta.csv", "raw.npy", "proc.npy", "g64.npy"]]
prev_cache = [Path(p).parent for p in glob.glob("/kaggle/input/**/fedigan_outputs/cache/meta.csv", recursive=True)]

if not all(f.exists() for f in cache_files) and prev_cache:
    for f in ["meta.csv", "raw.npy", "proc.npy", "g64.npy"]:
        shutil.copy(prev_cache[0] / f, CACHE / f)

if all(f.exists() for f in cache_files):
    META = pd.read_csv(CACHE / "meta.csv")
    RAW, PROC, G64 = (np.load(CACHE / f) for f in ["raw.npy", "proc.npy", "g64.npy"])
    print("Loaded preprocessing cache:", RAW.shape)
else:
    recs = []
    for c in CLASS_NAMES:
        for fn in sorted(os.listdir(os.path.join(DATA_ROOT, c))):
            recs.append((os.path.join(DATA_ROOT, c, fn), fn, c, CLASS_NAMES.index(c)))
    t0 = time.time()
    with Pool(NUM_WORKERS) as pool:
        outs = pool.map(_load_one, [r[0] for r in recs], chunksize=16)
    keep = [i for i, o in enumerate(outs) if o is not None]
    META = pd.DataFrame([dict(path=recs[i][0], file=recs[i][1], label_name=recs[i][2], label=recs[i][3],
                              md5=outs[i][0], dhash=outs[i][1]) for i in keep])
    RAW = np.stack([outs[i][2] for i in keep]); PROC = np.stack([outs[i][3] for i in keep])
    G64 = np.stack([outs[i][4] for i in keep])
    del outs; gc.collect()
    META.to_csv(CACHE / "meta.csv", index=False)
    for name, arr in [("raw", RAW), ("proc", PROC), ("g64", G64)]:
        np.save(CACHE / f"{name}.npy", arr)
    print(f"Preprocessed {len(META)} images in {time.time() - t0:.0f}s ({len(recs) - len(keep)} unreadable)")

META["arr_idx"] = np.arange(len(META))
LABELS_ALL = META["label"].to_numpy()
print(META["label_name"].value_counts().reindex(CLASS_NAMES))

# ---- exact duplicates (md5) ----
dup_report = []
drop = set()
for md5, g in META.groupby("md5"):
    if len(g) > 1:
        conflict = g["label"].nunique() > 1
        dup_report.append(dict(md5=md5, n=len(g), labels="|".join(sorted(g["label_name"].unique())), conflict=conflict))
        drop.update(g.index if conflict else g.index[1:])
DUP_DF = pd.DataFrame(dup_report)
DF = META.drop(index=sorted(drop)).reset_index(drop=True)

# ---- grouping: patient id (both eyes) + perceptual near-duplicates ----
parent = {}
def _find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb: parent[rb] = ra

pid_re = re.compile(r"^(\d+)_(left|right)\.(jpe?g|png)$", re.I)
for i, fn in enumerate(DF["file"]):
    node = f"img:{i}"; _find(node)
    m = pid_re.match(fn)
    if GROUP_SPLIT and m:
        _union(node, f"pid:{m.group(3).lower()}:{m.group(1)}")   # .jpg (ODIR) and .jpeg (EyePACS) ids are separate namespaces
if GROUP_SPLIT:
    for h, g in DF.groupby("dhash"):
        if 1 < len(g) <= 5:
            for j in g.index[1:]:
                _union(f"img:{g.index[0]}", f"img:{j}")
DF["group"] = [_find(f"img:{i}") for i in range(len(DF))]
n_near = int(sum(1 for _, g in DF.groupby("dhash") if 1 < len(g) <= 5))
print(f"Exact duplicate sets: {len(DUP_DF)} (label conflicts: {int(DUP_DF['conflict'].sum()) if len(DUP_DF) else 0}); "
      f"dropped {len(drop)} images | near-duplicate hash sets: {n_near} | groups: {DF['group'].nunique()} for {len(DF)} images")

def stratified_group_split(df, frac, seed):
    n_splits = max(2, int(round(1 / frac)))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    a, b = next(sgkf.split(df, df["label"], df["group"]))
    return df.iloc[a].reset_index(drop=True), df.iloc[b].reset_index(drop=True)

_trval, TEST_DF = stratified_group_split(DF, TEST_FRAC, SPLIT_SEED)
TRAIN_DF, VAL_DF = stratified_group_split(_trval, VAL_FRAC / (1 - TEST_FRAC), SPLIT_SEED)
assert not (set(TRAIN_DF.group) & set(TEST_DF.group)) and not (set(VAL_DF.group) & set(TEST_DF.group))
TRAIN_IDX, VAL_IDX, TEST_IDX = (d["arr_idx"].to_numpy() for d in (TRAIN_DF, VAL_DF, TEST_DF))

t01 = pd.DataFrame({
    "Class": SHORT,
    "Images (raw)": META["label"].value_counts().reindex(range(NUM_CLASSES), fill_value=0).values,
    "After de-dup": DF["label"].value_counts().reindex(range(NUM_CLASSES), fill_value=0).values,
    "Train": TRAIN_DF["label"].value_counts().reindex(range(NUM_CLASSES), fill_value=0).values,
    "Val": VAL_DF["label"].value_counts().reindex(range(NUM_CLASSES), fill_value=0).values,
    "Test": TEST_DF["label"].value_counts().reindex(range(NUM_CLASSES), fill_value=0).values})
t01.loc[len(t01)] = ["Total"] + t01.iloc[:, 1:].sum().tolist()
t01.to_csv(TBL / "T01_dataset.csv", index=False)
if len(DUP_DF): DUP_DF.to_csv(TBL / "T01b_duplicates.csv", index=False)
print(t01.to_string(index=False))

_PARTITIONS = {}
def get_partition(alpha):
    """Dirichlet split of TRAIN groups over clients; all images of a patient go to the same client."""
    if alpha in _PARTITIONS:
        return _PARTITIONS[alpha]
    gl = TRAIN_DF.groupby("group")["label"].agg(lambda s: s.value_counts().idxmax())
    for t in range(500):
        rng = np.random.default_rng(SPLIT_SEED + t)
        assign = {}
        for c in range(NUM_CLASSES):
            groups = gl[gl == c].index.to_numpy().copy(); rng.shuffle(groups)
            cuts = (np.cumsum(rng.dirichlet([alpha] * NUM_CLIENTS)) * len(groups)).astype(int)[:-1]
            for k, part in enumerate(np.split(groups, cuts)):
                for g in part: assign[g] = k
        client = TRAIN_DF["group"].map(assign).to_numpy()
        sizes = np.bincount(client, minlength=NUM_CLIENTS)
        if sizes.min() >= MIN_CLIENT_FRAC * len(TRAIN_DF):
            _PARTITIONS[alpha] = {k: TRAIN_IDX[client == k] for k in range(NUM_CLIENTS)}
            return _PARTITIONS[alpha]
    raise RuntimeError("Could not find a partition satisfying MIN_CLIENT_FRAC")

def synth_plan(counts):
    m = counts.max()
    return np.array([min(m - n, SYNTH_CAP) if n >= MIN_REAL_FOR_SYNTH else 0 for n in counts], dtype=int)

def partition_table(alpha):
    rows = []
    for k, idx in get_partition(alpha).items():
        cnt = np.bincount(LABELS_ALL[idx], minlength=NUM_CLASSES); plan = synth_plan(cnt)
        for c in range(NUM_CLASSES):
            rows.append(dict(alpha=alpha, client=k, cls=SHORT[c], real=int(cnt[c]), synthetic=int(plan[c]),
                             after=int(cnt[c] + plan[c]), few_shot_flag=bool(0 < cnt[c] < FEW_SHOT_WARN)))
    return pd.DataFrame(rows)

T02 = partition_table(ALPHA)
T02.to_csv(TBL / "T02_partition.csv", index=False)
print(T02.pivot(index="cls", columns="client", values=["real", "after"]).reindex(SHORT))
if T02.few_shot_flag.any():
    print("WARNING — classes with very few real images at a client (GAN output unreliable):\n",
          T02[T02.few_shot_flag][["client", "cls", "real"]].to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
w = 0.8 / NUM_CLIENTS
for k in range(NUM_CLIENTS):
    d = T02[T02.client == k]
    ax[0].bar(np.arange(NUM_CLASSES) + k * w, d.real, w, label=f"Client {k}")
    ax[1].bar(np.arange(NUM_CLASSES) + k * w, d.real, w, color=f"C{k}")
    ax[1].bar(np.arange(NUM_CLASSES) + k * w, d.synthetic, w, bottom=d.real, color=f"C{k}", alpha=0.35, hatch="//")
for a, t in zip(ax, [f"Real images per client (Dirichlet α={ALPHA})", "After cGAN augmentation (hatched = synthetic)"]):
    a.set_xticks(np.arange(NUM_CLASSES) + w * (NUM_CLIENTS - 1) / 2); a.set_xticklabels(SHORT); a.set_title(t); a.set_ylabel("Images")
top = (T02.real + T02.synthetic).max() * 1.1
for a in ax: a.set_ylim(0, top)
ax[0].legend(); plt.tight_layout(); plt.savefig(FIG / "F04_partition_and_augmentation.png", bbox_inches="tight"); plt.show()

def expected_calibration_error(y, prob, bins=15):
    conf, pred = prob.max(1), prob.argmax(1)
    edges = np.linspace(0, 1, bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): ece += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(ece)

def compute_metrics(y, prob):
    y, prob = np.asarray(y).astype(int), np.asarray(prob, dtype=float)
    pred = prob.argmax(1); n = len(y)
    present = sorted(set(y.tolist()))
    cm = confusion_matrix(y, pred, labels=list(range(NUM_CLASSES)))
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=present, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y, pred, labels=present, average="weighted", zero_division=0)
    out = dict(n=n, accuracy=accuracy_score(y, pred), balanced_accuracy=balanced_accuracy_score(y, pred),
               precision_macro=p, recall_macro=r, f1_macro=f, precision_weighted=pw, recall_weighted=rw, f1_weighted=fw,
               kappa=cohen_kappa_score(y, pred) if len(present) > 1 else float("nan"),
               mcc=matthews_corrcoef(y, pred) if len(present) > 1 else float("nan"),
               ece=expected_calibration_error(y, prob))
    per = {}
    aucs = []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]; fn = cm[c].sum() - tp; fp = cm[:, c].sum() - tp; tn = n - tp - fn - fp
        yb = (y == c).astype(int)
        auc_c = roc_auc_score(yb, prob[:, c]) if 0 < yb.sum() < n else float("nan")
        if not np.isnan(auc_c): aucs.append(auc_c)
        prec = tp / (tp + fp) if tp + fp else float("nan"); sens = tp / (tp + fn) if tp + fn else float("nan")
        per[SHORT[c]] = dict(support=int(tp + fn), precision=prec, recall_sensitivity=sens,
                             specificity=tn / (tn + fp) if tn + fp else float("nan"),
                             f1=2 * prec * sens / (prec + sens) if (tp and prec + sens) else 0.0,
                             ovr_accuracy=(tp + tn) / n, auc=auc_c,
                             ap=average_precision_score(yb, prob[:, c]) if 0 < yb.sum() < n else float("nan"))
    out["auc_macro"] = float(np.mean(aucs)) if aucs else float("nan")
    out = {k: float(v) for k, v in out.items()}
    out["per_class"] = per; out["cm"] = cm.tolist()
    return out

SCALARS = ["accuracy", "balanced_accuracy", "precision_macro", "recall_macro", "f1_macro",
           "precision_weighted", "recall_weighted", "f1_weighted", "auc_macro", "kappa", "mcc", "ece"]

def bootstrap_ci(y, prob, metrics=("accuracy", "f1_macro", "auc_macro", "kappa"), B=1000, seed=0):
    rng = np.random.default_rng(seed); y = np.asarray(y); prob = np.asarray(prob); vals = defaultdict(list)
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) < NUM_CLASSES: continue
        m = compute_metrics(y[i], prob[i])
        for k in metrics: vals[k].append(m[k])
    return {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for k, v in vals.items()}

def mean_sd_ci(v):
    v = np.asarray([x for x in v if not np.isnan(x)], float)
    if len(v) == 0: return (np.nan, np.nan, np.nan, np.nan, 0)
    if len(v) == 1: return (v[0], np.nan, np.nan, np.nan, 1)
    m, s = v.mean(), v.std(ddof=1); h = stats.t.ppf(0.975, len(v) - 1) * s / np.sqrt(len(v))
    return (m, s, m - h, m + h, len(v))

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(np.where(np.isnan(p), np.inf, p)); adj = np.full_like(p, np.nan)
    m = int((~np.isnan(p)).sum()); running = 0.0
    for rank, i in enumerate(order):
        if np.isnan(p[i]): continue
        running = max(running, min(1.0, (m - rank) * p[i])); adj[i] = running
    return adj

def mcnemar_exact(y, pred_a, pred_b):
    a_ok, b_ok = pred_a == y, pred_b == y
    b, c = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    p = stats.binomtest(min(b, c), b + c, 0.5).pvalue if b + c > 0 else 1.0
    return b, c, float(p)

def fmt(m, s, pct=True, d=2):
    if np.isnan(m): return "—"
    k = 100 if pct else 1
    return f"{m * k:.{d}f}" + ("" if np.isnan(s) else f" ± {s * k:.{d}f}")

class ArrayDS(Dataset):
    def __init__(self, src, idx, labels, tf):
        self.src, self.idx, self.labels, self.tf = src, np.asarray(idx), np.asarray(labels), tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        x = tv_tensors.Image(torch.from_numpy(np.ascontiguousarray(self.src[self.idx[i]])).permute(2, 0, 1))
        return self.tf(x), int(self.labels[i])

def make_tfs(mean, std):
    train = T.Compose([T.RandomHorizontalFlip(), T.RandomRotation(10), T.ColorJitter(0.15, 0.15, 0.15),
                       T.ToDtype(torch.float32, scale=True), T.Normalize(mean, std), T.ToPureTensor()])
    test = T.Compose([T.ToDtype(torch.float32, scale=True), T.Normalize(mean, std), T.ToPureTensor()])
    return train, test

def train_loader(parts, tf, seed):
    ds = ConcatDataset([ArrayDS(src, idx, lab, tf) for src, idx, lab in parts if len(idx)])
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=len(ds) > BATCH, num_workers=NUM_WORKERS,
                      pin_memory=True, generator=g, persistent_workers=NUM_WORKERS > 0)

def eval_loader(idx, tf):
    return DataLoader(ArrayDS(PROC, idx, LABELS_ALL[idx], tf), batch_size=64, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)

_INIT = {}
def new_model(backbone, seed=None):
    if backbone not in _INIT:
        m = timm.create_model(backbone, pretrained=PRETRAINED, num_classes=NUM_CLASSES, drop_rate=DROP_RATE)
        _INIT[backbone] = ({k: v.detach().clone() for k, v in m.state_dict().items()},
                           timm.data.resolve_model_data_config(m))
        del m
    m = timm.create_model(backbone, pretrained=False, num_classes=NUM_CLASSES, drop_rate=DROP_RATE)
    m.load_state_dict(_INIT[backbone][0])
    if seed is not None:
        torch.manual_seed(seed); m.reset_classifier(NUM_CLASSES)   # seed-dependent head initialisation
    return m.to(DEVICE)

def lr_for(backbone): return LR["vit"] if "vit" in backbone else LR["cnn"]

class ClassBalancedFocalLoss(nn.Module):
    """Cui et al. (2019) class-balanced weights + focal term (Lin et al., 2017). Weights from LOCAL counts."""
    def __init__(self, counts, beta=CB_BETA, gamma=FOCAL_GAMMA):
        super().__init__()
        counts = np.asarray(counts, float); present = counts > 0
        w = np.zeros_like(counts)
        w[present] = (1 - beta) / (1 - np.power(beta, counts[present]))
        w[present] = w[present] / w[present].sum() * present.sum()
        w[~present] = 1.0
        self.register_buffer("w", torch.tensor(w, dtype=torch.float32)); self.gamma = gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits, 1).gather(1, y[:, None]).squeeze(1)
        return (self.w[y] * (1 - logp.exp()) ** self.gamma * (-logp)).mean()

class ConditionalGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT, num_classes=NUM_CLASSES, embed_dim=EMBED):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, embed_dim)
        self.fc = nn.Sequential(nn.Linear(latent_dim + embed_dim, 256 * 8 * 8), nn.ReLU(True))
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),   # 16
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),     # 32
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),      # 64
            nn.Conv2d(32, 3, 3, 1, 1), nn.Tanh())
    def forward(self, z, y):
        return self.net(self.fc(torch.cat([z, self.label_emb(y)], 1)).view(-1, 256, 8, 8))

class ConditionalDiscriminator(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embed_dim=EMBED):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, embed_dim)
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True))
        self.class_proj = nn.Linear(embed_dim, 64)
        self.final = nn.Sequential(nn.Conv2d(257, 128, 3, 1, 1), nn.LeakyReLU(0.2, True), nn.Flatten(),
                                   nn.Linear(128 * 8 * 8, 1), nn.Sigmoid())
    def forward(self, x, y):
        return self.final(torch.cat([self.net(x), self.class_proj(self.label_emb(y)).view(-1, 1, 8, 8)], 1))

def train_cgan(x64, labels, seed, epochs=GAN_EPOCHS):
    set_seed(seed)
    X = torch.from_numpy(x64).permute(0, 3, 1, 2).float().div(127.5).sub(1).to(DEVICE)
    Y = torch.from_numpy(labels.astype(np.int64)).to(DEVICE)
    G, D = ConditionalGenerator().to(DEVICE), ConditionalDiscriminator().to(DEVICE)
    oG = torch.optim.Adam(G.parameters(), GAN_LR, betas=(GAN_BETA1, 0.999))
    oD = torch.optim.Adam(D.parameters(), GAN_LR, betas=(GAN_BETA1, 0.999))
    bce = nn.BCELoss(); n = len(X); bs = min(GAN_BATCH, n); hist = {"g": [], "d": []}
    for ep in range(epochs):
        perm = torch.randperm(n, device=DEVICE); gs, ds_, nb = 0.0, 0.0, 0
        for i in range(0, n - bs + 1, bs):
            b = perm[i:i + bs]; real, y = X[b], Y[b]
            flip = torch.rand(bs, device=DEVICE) < 0.5
            real = torch.where(flip[:, None, None, None], real.flip(3), real)
            fake = G(torch.randn(bs, LATENT, device=DEVICE), y)
            dl = 0.5 * (bce(D(real, y), torch.full((bs, 1), 0.9, device=DEVICE)) +
                        bce(D(fake.detach(), y), torch.zeros(bs, 1, device=DEVICE)))
            oD.zero_grad(); dl.backward(); oD.step()
            gl = bce(D(fake, y), torch.ones(bs, 1, device=DEVICE))
            oG.zero_grad(); gl.backward(); oG.step()
            gs += gl.item(); ds_ += dl.item(); nb += 1
        hist["g"].append(gs / max(nb, 1)); hist["d"].append(ds_ / max(nb, 1))
    return G.eval(), hist

@torch.no_grad()
def generate(G, plan, seed):
    g = torch.Generator(device=DEVICE); g.manual_seed(seed)
    ys = np.concatenate([np.full(n, c) for c, n in enumerate(plan) if n > 0]).astype(np.int64) if plan.sum() else np.zeros(0, np.int64)
    imgs = []
    for i in range(0, len(ys), 128):
        y = torch.from_numpy(ys[i:i + 128]).to(DEVICE)
        x = G(torch.randn(len(y), LATENT, device=DEVICE, generator=g), y)
        imgs.append(((x * 0.5 + 0.5).clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1).cpu().numpy())
    return (np.concatenate(imgs) if imgs else np.zeros((0, GAN_SIZE, GAN_SIZE, 3), np.uint8)), ys

_SYN = {}
def get_synthetic(seed, alpha=ALPHA):
    """Per-client cGAN trained only on that client's real images. Returns {k: (imgs224, labels, imgs64)}."""
    key = (seed, alpha)
    if key in _SYN: return _SYN[key]
    path = OUT / "synthetic" / f"syn_a{alpha}_s{seed}_{PROFILE}.npz"
    out = {}
    if path.exists():
        z = np.load(path)
        for k in range(NUM_CLIENTS): out[k] = (z[f"x224_{k}"], z[f"y_{k}"], z[f"x64_{k}"])
    else:
        save, hists = {}, {}
        for k, idx in get_partition(alpha).items():
            lab = LABELS_ALL[idx]
            G, hist = train_cgan(G64[idx], lab, seed * 10 + k)
            torch.save(G.state_dict(), OUT / "gan" / f"G_a{alpha}_s{seed}_c{k}_{PROFILE}.pt")
            x64, ys = generate(G, synth_plan(np.bincount(lab, minlength=NUM_CLASSES)), seed * 10 + k)
            x224 = np.stack([cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC) for im in x64]) \
                if len(x64) else np.zeros((0, IMG_SIZE, IMG_SIZE, 3), np.uint8)
            out[k] = (x224, ys, x64); hists[k] = hist
            save.update({f"x224_{k}": x224, f"y_{k}": ys, f"x64_{k}": x64})
            del G; torch.cuda.empty_cache()
        np.savez_compressed(path, **save)
        json.dump(hists, open(OUT / "gan" / f"loss_a{alpha}_s{seed}_{PROFILE}.json", "w"))
    _SYN[key] = out
    return out

def client_units(seed, aug, alpha=ALPHA):
    part = get_partition(alpha)
    syn = get_synthetic(seed, alpha) if aug == "gan" else None
    units = []
    for k in range(NUM_CLIENTS):
        idx = part[k]; lab = LABELS_ALL[idx]; cnt = np.bincount(lab, minlength=NUM_CLASSES)
        plan = synth_plan(cnt); parts = [(PROC, idx, lab)]
        if aug == "gan":
            x224, ys, _ = syn[k]
            parts.append((x224, np.arange(len(ys)), ys)); cnt = cnt + np.bincount(ys, minlength=NUM_CLASSES)
        elif aug == "oversample":
            rng = np.random.default_rng(seed * 100 + k)
            extra = np.concatenate([rng.choice(idx[lab == c], n, replace=True) for c, n in enumerate(plan) if n > 0] or [np.zeros(0, int)])
            parts.append((PROC, extra, LABELS_ALL[extra])); cnt = cnt + plan
        units.append(dict(client=k, parts=parts, counts=cnt, n_real=len(idx), real_idx=idx))
    return units

def merge_units(units):
    return dict(client="all", parts=[p for u in units for p in u["parts"]], counts=sum(u["counts"] for u in units),
                n_real=sum(u["n_real"] for u in units), real_idx=np.concatenate([u["real_idx"] for u in units]))

def _dp(model): return nn.DataParallel(model) if N_GPU > 1 else model

def train_local(model, loader, crit, lr, epochs, prox_ref=None, mu=0.0):
    net = _dp(model); model.train()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    tot, n, correct = 0.0, 0, 0
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out = net(x)
            loss = crit(out.float(), y)
            if prox_ref is not None and mu > 0:
                loss = loss + 0.5 * mu * sum(((p - q) ** 2).sum() for p, q in zip(params, prox_ref))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y); n += len(y); correct += (out.argmax(1) == y).sum().item()
    return tot / max(n, 1), correct / max(n, 1)

@torch.no_grad()
def predict(model, loader):
    net = _dp(model); model.eval(); P, Y = [], []
    for x, y in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out = net(x.to(DEVICE, non_blocking=True))
        P.append(F.softmax(out.float(), 1).cpu()); Y.append(y)
    return torch.cat(Y).numpy(), torch.cat(P).numpy()

def nll(y, p): return float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-8, 1)).mean())

def aggregate(states, weights):
    w = np.asarray(weights, float); w = w / w.sum(); new = {}
    for k, v0 in states[0].items():
        new[k] = sum(float(wi) * s[k].float() for wi, s in zip(w, states)).to(v0.dtype) if v0.is_floating_point() else v0.clone()
    return new

def fl_train(units, backbone, seed, agg, mu, participation, tf_train, val_ld, test_ld, tag=""):
    set_seed(seed)
    gm = new_model(backbone, seed); local = new_model(backbone)
    lr0 = lr_for(backbone); rng = np.random.default_rng(seed)
    crits = [ClassBalancedFocalLoss(u["counts"]).to(DEVICE) for u in units]
    loaders = [train_loader(u["parts"], tf_train, seed * 1000 + i) for i, u in enumerate(units)]
    n_params = sum(p.numel() for p in gm.parameters()); comm = 0.0
    hist, best, best_state, best_round, bad = [], -1.0, None, 0, 0
    for r in range(1, ROUNDS + 1):
        lr = lr0 * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * (r - 1) / max(ROUNDS, 1))))
        chosen = [i for i in range(len(units)) if rng.random() < participation] if participation < 1 else list(range(len(units)))
        if not chosen: chosen = [int(rng.integers(len(units)))]
        gstate = {k: v.detach().clone() for k, v in gm.state_dict().items()}
        prox_ref = [p.detach().clone() for p in gm.parameters() if p.requires_grad] if agg == "fedprox" else None
        states, weights, tl, ta = [], [], [], []
        for i in chosen:
            local.load_state_dict(gstate)
            l, a = train_local(local, loaders[i], crits[i], lr, LOCAL_EPOCHS, prox_ref, mu)
            states.append({k: v.detach().clone() for k, v in local.state_dict().items()})
            weights.append(units[i]["n_real"] if AGG_WEIGHT == "real" else len(loaders[i].dataset))
            tl.append(l); ta.append(a)
        gm.load_state_dict(aggregate(states, weights) if len(states) > 1 else states[0])
        comm += 2 * len(chosen) * n_params * 4 / 1e6
        del states; torch.cuda.empty_cache()
        yv, pv = predict(gm, val_ld); yt, pt = predict(gm, test_ld)
        mv, mt = compute_metrics(yv, pv), compute_metrics(yt, pt)
        hist.append(dict(round=r, lr=lr, participants=len(chosen), train_loss=float(np.mean(tl)), train_acc=float(np.mean(ta)),
                         val_loss=nll(yv, pv), val_acc=mv["accuracy"], val_f1=mv["f1_macro"], val_auc=mv["auc_macro"],
                         test_loss=nll(yt, pt), test_acc=mt["accuracy"], test_f1=mt["f1_macro"], test_auc=mt["auc_macro"]))
        print(f"  {tag} r{r:02d} | clients {chosen} | train loss {hist[-1]['train_loss']:.3f} acc {hist[-1]['train_acc']:.3f}"
              f" | val acc {mv['accuracy']:.4f} auc {mv['auc_macro']:.4f} | test acc {mt['accuracy']:.4f}")
        score = float(np.nan_to_num(mv["auc_macro"], nan=0.0)) + 1e-3 * mv["accuracy"]
        if score > best:
            best, best_round, bad = score, r, 0
            best_state = {k: v.detach().cpu().clone() for k, v in gm.state_dict().items()}
        else:
            bad += 1
            if r >= MIN_ROUNDS and bad >= PATIENCE: break
    gm.load_state_dict(best_state); del local; torch.cuda.empty_cache()
    return gm, hist, best_round, comm

def run_experiment(exp, seed):
    t0 = time.time(); backbone = exp.get("backbone", BACKBONE); alpha = exp.get("alpha", ALPHA)
    mu = exp.get("mu", MU) if exp["agg"] == "fedprox" else 0.0
    new_model(backbone)  # make sure init weights / data config are cached
    tf_train, tf_test = make_tfs(_INIT[backbone][1]["mean"], _INIT[backbone][1]["std"])
    val_ld, test_ld = eval_loader(VAL_IDX, tf_test), eval_loader(TEST_IDX, tf_test)
    units = client_units(seed, exp["aug"], alpha)
    holdout = exp.get("holdout")
    res = dict(key=exp["key"], name=exp["name"], group=exp["group"], seed=seed, mode=exp["mode"], backbone=backbone,
               agg=exp["agg"], aug=exp["aug"], mu=mu, alpha=alpha, participation=exp.get("participation", 1.0),
               holdout=holdout, profile=PROFILE)
    if exp["mode"] == "local":
        per, probs = [], []
        for u in units:
            m, hist, br, comm = fl_train([u], backbone, seed, "fedavg", 0.0, 1.0, tf_train, val_ld, test_ld, f"[{exp['key']} c{u['client']}]")
            y, p = predict(m, test_ld); per.append(dict(client=u["client"], **{k: v for k, v in compute_metrics(y, p).items() if k in SCALARS}))
            probs.append(p); del m
        test = {k: float(np.mean([d[k] for d in per])) for k in SCALARS}
        res.update(test=test, per_client=per, history=hist, best_round=br, comm_MB=0.0)
        np.savez_compressed(RUNS_DIR / f"{exp['key']}__s{seed}.npz", y=y, prob=np.stack(probs))
    else:
        if exp["mode"] == "central":
            train_units = [merge_units(units)]
        else:
            train_units = [u for u in units if u["client"] != holdout]
        m, hist, br, comm = fl_train(train_units, backbone, seed, exp["agg"], mu, exp.get("participation", 1.0),
                                     tf_train, val_ld, test_ld, f"[{exp['key']} s{seed}]")
        y, p = predict(m, test_ld)
        res.update(test=compute_metrics(y, p), history=hist, best_round=br,
                   comm_MB=0.0 if exp["mode"] == "central" else comm)
        np.savez_compressed(RUNS_DIR / f"{exp['key']}__s{seed}.npz", y=y, prob=p)
        if holdout is not None:
            hidx = units[holdout]["real_idx"]; yu, pu = predict(m, eval_loader(hidx, tf_test))
            res["unseen_client"] = compute_metrics(yu, pu)
            np.savez_compressed(RUNS_DIR / f"{exp['key']}__s{seed}__unseen.npz", y=yu, prob=pu)
        if exp["key"] == "proposed" and seed == SEEDS_MAIN[0]:
            torch.save(m.state_dict(), OUT / "models" / f"proposed_s{seed}_{RUN_TAG}.pth")
        del m
    res["train_time_s"] = time.time() - t0
    json.dump(res, open(RUNS_DIR / f"{exp['key']}__s{seed}.json", "w"), default=float)
    gc.collect(); torch.cuda.empty_cache()
    return res

def E(key, name, group, mode="fed", agg="fedprox", aug="gan", seeds="other", **kw):
    return dict(key=key, name=name, group=group, mode=mode, agg=agg, aug=aug, seeds=seeds, **kw)

EXPERIMENTS = [
    E("proposed", "FediGAN-ViT (FedProx + cGAN + ViT)", "main", seeds="main"),
    # --- ablation: components ---
    E("fedavg_gan", "FedAvg + cGAN + ViT", "ablation", agg="fedavg"),
    E("fedprox_none", "FedProx + ViT (no augmentation)", "ablation", aug="none"),
    E("fedavg_none", "FedAvg + ViT (no augmentation)", "ablation", agg="fedavg", aug="none"),
    E("fedprox_oversample", "FedProx + ViT + random oversampling (same counts as cGAN)", "ablation", aug="oversample"),
    E("local_gan", "Local-only ViT + cGAN (no collaboration, mean of clients)", "ablation", mode="local", agg="fedavg"),
    E("central_none", "Centralised ViT (pooled data, upper bound)", "ablation", mode="central", agg="fedavg", aug="none"),
    E("central_gan", "Centralised ViT + cGAN", "ablation", mode="central", agg="fedavg"),
    # --- backbones ---
    E("fedprox_gan_resnet50", "FedProx + cGAN + ResNet-50", "backbone", backbone="resnet50"),
    E("fedprox_gan_effb0", "FedProx + cGAN + EfficientNet-B0", "backbone", backbone="efficientnet_b0"),
    # --- robustness ---
    E("partial", f"Partial participation (p={PARTICIPATION})", "robustness", participation=PARTICIPATION),
    *[E(f"loco_c{k}", f"Leave client {k} out (unseen institution)", "robustness", holdout=k) for k in range(NUM_CLIENTS)],
    # --- sensitivity (lowest priority) ---
    *[E(f"alpha_{a}", f"Dirichlet α = {a}", "sensitivity", alpha=a) for a in SENS_ALPHAS],
    *[E(f"mu_{m}", f"FedProx μ = {m}", "sensitivity", mu=m) for m in SENS_MUS],
]
EXP_BY_KEY = {e["key"]: e for e in EXPERIMENTS}

PLAN = []
for s in SEEDS_OTHER:                                   # every core config on each shared seed first
    PLAN += [(e, s) for e in EXPERIMENTS if e["group"] != "sensitivity"]
PLAN += [(EXP_BY_KEY["proposed"], s) for s in SEEDS_MAIN if s not in SEEDS_OTHER]
for s in SEEDS_OTHER:
    PLAN += [(e, s) for e in EXPERIMENTS if e["group"] == "sensitivity"]
if PLAN_MODE == "minimal":
    KEY3 = ["proposed", "fedprox_none", "fedprox_oversample", "local_gan", "central_none"]   # 3 seeds
    ONE = ["fedavg_gan", "partial", "loco_c0", "fedprox_gan_resnet50"]                           # 1 seed
    PLAN = [(EXP_BY_KEY[k], s) for s in SEEDS_OTHER for k in KEY3] + [(EXP_BY_KEY[k], SEEDS_OTHER[0]) for k in ONE]
elif PLAN_MODE == "reduced":
    KEY3 = ["proposed", "fedprox_none", "fedprox_oversample", "local_gan", "central_none",
            "partial", "loco_c0", "loco_c1", "loco_c2"]          # 3 seeds: needed for paired statistics
    ONE = ["fedavg_gan", "fedavg_none", "central_gan", "fedprox_gan_resnet50", "fedprox_gan_effb0"]  # 1 seed
    PLAN = [(EXP_BY_KEY[k], s) for s in SEEDS_OTHER for k in KEY3] + [(EXP_BY_KEY[k], SEEDS_OTHER[0]) for k in ONE]
print(f"{len(PLAN)} runs planned ({PLAN_MODE} plan)")

# ============================== BACKGROUND WORKER ROLES ==============================
_ROLE = os.environ.get("FEDIGAN_ROLE")
if _ROLE == "prep":
    for bb in sorted({e.get("backbone", BACKBONE) for e, _ in PLAN}):
        new_model(bb); print("backbone ready:", bb, flush=True)
    need = sorted({(s, e.get("alpha", ALPHA)) for e, s in PLAN if e["aug"] == "gan"})
    for s, a in need:
        t = time.time(); get_synthetic(s, a); print(f"cGANs ready: seed {s}, alpha {a} ({time.time() - t:.0f}s)", flush=True)
    print("PREP DONE", flush=True)
elif _ROLE == "shard":
    todo = [(e, s) for e, s in PLAN if not (RUNS_DIR / f"{e['key']}__s{s}.json").exists()]
    shard, nshards = int(_SHARD), int(os.environ.get("FEDIGAN_NSHARDS", 1))
    mine = todo[shard::nshards]
    print(f"[GPU{shard}] {len(mine)} of {len(todo)} remaining experiments", flush=True)
    durations, skipped = [], []
    for i, (exp, seed) in enumerate(mine):
        if (RUNS_DIR / f"{exp['key']}__s{seed}.json").exists():
            continue
        est = (np.median(durations) * 1.3) if durations else 1500
        if time.time() - NOTEBOOK_START + est > TIME_BUDGET_H * 3600:
            skipped.append(f"{exp['key']}__s{seed}"); continue
        print(f"\n[GPU{shard}] [{i + 1}/{len(mine)}] {exp['name']} | seed {seed} | "
              f"session time {(time.time() - NOTEBOOK_START) / 3600:.2f} h", flush=True)
        r = run_experiment(exp, seed); durations.append(r["train_time_s"])
        print(f"[GPU{shard}]  -> test acc {r['test']['accuracy']:.4f} | macro-F1 {r['test']['f1_macro']:.4f} | "
              f"macro-AUC {r['test']['auc_macro']:.4f} | {r['train_time_s'] / 60:.1f} min", flush=True)
    json.dump(dict(skipped=skipped), open(OUT / f"skipped_{RUN_TAG}_shard{shard}.json", "w"))
    print(f"[GPU{shard}] DONE. skipped for time: {skipped}", flush=True)


In [ ]:
# ============================== RESTORE PREVIOUS RESULTS (if a previous output is attached) ==============================
import glob, shutil
from pathlib import Path
OUTP = Path("/kaggle/working/fedigan_outputs"); restored = 0
sources = {f"runs_{RUN_TAG}": f"/kaggle/input/**/runs_{RUN_TAG}/*",
           "cache": "/kaggle/input/**/fedigan_outputs/cache/*", "synthetic": "/kaggle/input/**/fedigan_outputs/synthetic/*",
           "gan": "/kaggle/input/**/fedigan_outputs/gan/*", "models": "/kaggle/input/**/fedigan_outputs/models/*"}
for sub, pattern in sources.items():
    (OUTP / sub).mkdir(parents=True, exist_ok=True)
    for p in glob.glob(pattern, recursive=True):
        dst = OUTP / sub / os.path.basename(p)
        if os.path.isfile(p) and not dst.exists():
            shutil.copy(p, dst); restored += 1
done = len(glob.glob(str(OUTP / f"runs_{RUN_TAG}" / "*__s[0-9][0-9].json")))
print(f"Restored {restored} files | finished experiments available: {done}")

## 2 · Train all experiments (two GPUs in parallel)

In [ ]:
# ============================== RUN ALL EXPERIMENTS: one background worker per GPU ==============================
# Safe to re-run: if workers are already running it only monitors. Interrupting this cell does NOT stop the workers.
import subprocess
W = "/kaggle/working"; ENGINE = f"{W}/fedigan_core.py"
base = dict(os.environ, FEDIGAN_WORKERS="2", PYTHONUNBUFFERED="1")
n_gpu = len([l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if l.startswith("GPU")])
assert n_gpu >= 1, "No GPU found - set Accelerator to GPU T4 x2"

def tail(name, n=1):
    try:
        lines = [l for l in open(f"{W}/{name}").read().splitlines() if l.strip()]
        return "\n      ".join(l[:140] for l in lines[-n:]) or "(starting)"
    except FileNotFoundError:
        return "(no log yet)"
def gpus():
    return subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip().replace("\n", " | ")
def workers():
    """Count running engine processes by reading /proc (never counts this check itself)."""
    n = 0
    for f in glob.glob("/proc/[0-9]*/cmdline"):
        try:
            argv = open(f, "rb").read().split(b"\0")
        except OSError:
            continue
        if len(argv) > 1 and argv[1].endswith(b"fedigan_core.py"):
            n += 1
    return n
def finished():
    return len(glob.glob(f"{W}/fedigan_outputs/runs_{RUN_TAG}/*__s[0-9][0-9].json"))

if workers() == 0:
    print("Step 1/2: data cache, backbone downloads and client cGANs (first time ~10-20 min) ...", flush=True)
    p = subprocess.Popen(["python", ENGINE], env=dict(base, FEDIGAN_ROLE="prep", CUDA_VISIBLE_DEVICES="0"),
                         stdout=open(f"{W}/prep.log", "w"), stderr=subprocess.STDOUT)
    while p.poll() is None:
        time.sleep(30); print(time.strftime("%H:%M"), "| prep |", tail("prep.log"), flush=True)
    print(tail("prep.log", 8))
    assert p.returncode == 0, "Preparation failed - see the log above (full log: /kaggle/working/prep.log)"
    print(f"\nStep 2/2: training on {n_gpu} GPU(s) in parallel", flush=True)
    for g in range(n_gpu):
        subprocess.Popen(["python", ENGINE], stdout=open(f"{W}/gpu{g}.log", "w"), stderr=subprocess.STDOUT, start_new_session=True,
                         env=dict(base, FEDIGAN_ROLE="shard", CUDA_VISIBLE_DEVICES=str(g), FEDIGAN_SHARD=str(g), FEDIGAN_NSHARDS=str(n_gpu)))
    time.sleep(90)
else:
    print("Workers already running - monitoring only.")

while workers() > 0:
    print(f"{time.strftime('%H:%M')} | finished experiments: {finished()} | GPU util/mem: {gpus()}", flush=True)
    for g in range(n_gpu):
        print(f"   GPU{g}: {tail(f'gpu{g}.log')}", flush=True)
    time.sleep(120)
for g in range(n_gpu):
    print(f"GPU{g} final:\n      {tail(f'gpu{g}.log', 4)}")
print(f"\nAll workers finished. Finished experiments: {finished()}")

## 3 · Analysis — runs after training finishes

In [ ]:
# ============================== LOAD THE ENGINE INTO THIS NOTEBOOK (for analysis) ==============================
%run /kaggle/working/fedigan_core.py

### Data manifests
Writes the patient-group / split / client assignment of every image (`split_manifest.csv`), the per-client synthetic counts (`augmentation_manifest.csv`) and the duplicate/hash settings (`hash_settings.json`) to `fedigan_outputs/manifests/`. Needs no GPU and no training.

In [ ]:
# ============================== EXPORT DATA MANIFESTS (audit files; no training, no GPU needed) ==============================
# Writes fedigan_outputs/manifests/: split_manifest.csv (one row per image), augmentation_manifest.csv and hash_settings.json.
MAN = OUT / "manifests"; MAN.mkdir(parents=True, exist_ok=True)
split_of = {int(a): name for name, d in [("train", TRAIN_DF), ("val", VAL_DF), ("test", TEST_DF)] for a in d["arr_idx"]}
client_of = {int(a): int(k) for k, idx in get_partition(ALPHA).items() for a in idx}
group_of = dict(zip(DF["arr_idx"].astype(int), DF["group"]))
gid = {g: i for i, g in enumerate(sorted(DF["group"].unique()))}

def _hex64(v):                                   # dHash bits -> 16-digit hex (cache may store bits as str or int)
    return f"{int(str(v).zfill(64), 2):016x}"

def _pid(fn):
    m = pid_re.match(fn)
    return f"{m.group(3).lower()}:{m.group(1)}" if m else ""

man = pd.DataFrame({
    "file": META["file"], "class": META["label_name"], "md5": META["md5"], "dhash_hex": META["dhash"].map(_hex64),
    "patient_id_from_filename": META["file"].map(_pid),
    "group_id": META["arr_idx"].map(lambda a: gid[group_of[a]] if a in group_of else -1),
    "split": META["arr_idx"].map(lambda a: split_of.get(int(a), "excluded_conflicting_duplicate")),
    "client": META["arr_idx"].map(lambda a: client_of.get(int(a), "")),
})
man.to_csv(MAN / "split_manifest.csv", index=False)

aug = []
for k, idx in get_partition(ALPHA).items():
    cnt = np.bincount(LABELS_ALL[idx], minlength=NUM_CLASSES); plan = synth_plan(cnt)
    for c in range(NUM_CLASSES):
        aug.append(dict(client=k, cls=SHORT[c], real=int(cnt[c]), synthetic=int(plan[c]), total=int(cnt[c] + plan[c]),
                        synthetic_noise_seed="10*seed + client (seeds 42, 43, 44)"))
pd.DataFrame(aug).to_csv(MAN / "augmentation_manifest.csv", index=False)

json.dump({"exact_duplicates": "MD5 of file bytes; all copies of a duplicate set with conflicting labels are removed",
           "perceptual_hash": "dHash, 9x8 greyscale thumbnail of the 224x224 image, 64 bits",
           "near_duplicate_rule": "identical hash (Hamming distance 0), linked only for hash sets of 2-5 images",
           "patient_rule": "file name <id>_left/<id>_right; ids separated by file extension (.jpg vs .jpeg)",
           "grouping": "union-find over patient and hash links, before any splitting",
           "split": f"StratifiedGroupKFold, test {TEST_FRAC}, val {VAL_FRAC}, seed {SPLIT_SEED}",
           "partition": f"group-level Dirichlet alpha={ALPHA}, {NUM_CLIENTS} clients, each >= {MIN_CLIENT_FRAC:.0%} of training data"},
          open(MAN / "hash_settings.json", "w"), indent=1)

# audit: every group lies in exactly one split, and every training group in exactly one client
used = man[man.group_id >= 0]
assert used.groupby("group_id")["split"].nunique().max() == 1, "a group crosses splits"
tr = used[used.split == "train"]
assert tr.groupby("group_id")["client"].nunique().max() == 1, "a group crosses clients"
print(man["split"].value_counts().to_string(), f"\ngroups: {used.group_id.nunique()} | manifests written to {MAN}")

In [ ]:
# ---- preprocessing illustration (Fig. 3) ----
fig, ax = plt.subplots(2, 4, figsize=(11, 6.6))
for j, c in enumerate(range(NUM_CLASSES)):
    i = TRAIN_DF[TRAIN_DF.label == c]["arr_idx"].iloc[0]
    ax[0, j].imshow(RAW[i]); ax[0, j].set_title(f"{SHORT[c]} — original"); ax[0, j].axis("off")
    ax[1, j].imshow(PROC[i]); ax[1, j].set_title("Preprocessed"); ax[1, j].axis("off")
plt.suptitle("Illumination correction + NLM denoising + CLAHE"); plt.tight_layout()
plt.savefig(OUT / "figures/F03_preprocessing.png", bbox_inches="tight"); plt.show()

## 4 · cGAN quality (FID, KID, IS, noise floor, memorisation) — seed 42

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.inception import InceptionScore

def _to_t(x): return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous()

def inception_metrics(real64, fake64):
    out = {}
    try:
        fid = FrechetInceptionDistance(feature=2048, normalize=False).to(DEVICE)
        kid = KernelInceptionDistance(subset_size=max(2, min(100, len(real64), len(fake64)) - 1), normalize=False).to(DEVICE)
        for i in range(0, len(real64), 64):
            b = _to_t(real64[i:i + 64]).to(DEVICE); fid.update(b, real=True); kid.update(b, real=True)
        for i in range(0, len(fake64), 64):
            b = _to_t(fake64[i:i + 64]).to(DEVICE); fid.update(b, real=False); kid.update(b, real=False)
        out["FID"] = float(fid.compute()); km, ks = kid.compute(); out["KID_x1e3"] = float(km) * 1e3; out["KID_sd_x1e3"] = float(ks) * 1e3
    except Exception as ex:
        print("FID/KID failed:", ex)
    return out

def nn_dist(a, b):
    A = a.reshape(len(a), -1).astype(np.float32) / 255; B = torch.from_numpy(b.reshape(len(b), -1).astype(np.float32) / 255).to(DEVICE)
    d = []
    for i in range(0, len(A), 256):
        d.append(torch.cdist(torch.from_numpy(A[i:i + 256]).to(DEVICE), B).min(1).values.cpu())
    return torch.cat(d).numpy()

s0 = SEEDS_MAIN[0]
syn = get_synthetic(s0, ALPHA)
gan_rows = []
for k, idx in get_partition(ALPHA).items():
    real = G64[idx]; lab = LABELS_ALL[idx]; rng = np.random.default_rng(0)
    G = ConditionalGenerator().to(DEVICE); G.load_state_dict(torch.load(OUT / "gan" / f"G_a{ALPHA}_s{s0}_c{k}_{PROFILE}.pt")); G.eval()
    n_fake = max(len(real), 500)
    plan = np.bincount(rng.choice(lab, n_fake), minlength=NUM_CLASSES)     # same class mix as the client's real data
    fake, _ = generate(G, plan, 123 + k)
    row = dict(client=k, n_real=len(real), n_fake=len(fake), **inception_metrics(real, fake))
    perm = rng.permutation(len(real)); h = len(real) // 2
    floor = inception_metrics(real[perm[:h]], real[perm[h:]])
    row["FID_real_vs_real_floor"] = floor.get("FID", np.nan)
    try:
        isc = InceptionScore(normalize=False).to(DEVICE)
        for i in range(0, len(fake), 64): isc.update(_to_t(fake[i:i + 64]).to(DEVICE))
        m_, s_ = isc.compute(); row["IS"] = float(m_); row["IS_sd"] = float(s_)
    except Exception as ex:
        print("IS failed:", ex)
    d_fake = nn_dist(fake[:500], real); d_test = nn_dist(G64[TEST_IDX][:500], real)
    row["NN_dist_fake_median"] = float(np.median(d_fake)); row["NN_dist_test_median"] = float(np.median(d_test))
    row["memorisation_ratio"] = row["NN_dist_fake_median"] / row["NN_dist_test_median"]   # <<1 suggests copying
    gan_rows.append(row); print(row)

    # real vs synthetic at native 64x64
    x64, ys = syn[k][2], syn[k][1]
    fig, ax = plt.subplots(NUM_CLASSES, 12, figsize=(14, 5.2))
    for c in range(NUM_CLASSES):
        r_i = idx[lab == c][:6]; f_i = np.where(ys == c)[0][:6]
        for j in range(12):
            a = ax[c, j]; a.axis("off")
            if j < 6 and j < len(r_i): a.imshow(G64[r_i[j]])
            if j >= 6 and j - 6 < len(f_i): a.imshow(x64[f_i[j - 6]])
        ax[c, 0].set_title(f"{SHORT[c]} (real n={int((lab == c).sum())})", fontsize=8, loc="left")
    fig.text(0.27, 0.99, "Real (64×64)", ha="center"); fig.text(0.73, 0.99, "cGAN synthetic (64×64)", ha="center")
    plt.tight_layout(); plt.savefig(OUT / f"figures/F07_real_vs_synthetic_client{k}.png", bbox_inches="tight"); plt.show()

    # memorisation check: synthetic images and their nearest real training image
    order = np.argsort(d_fake)[:8]
    fig, ax = plt.subplots(2, 8, figsize=(12, 3.4))
    Rf = real.reshape(len(real), -1).astype(np.float32)
    for j, i in enumerate(order):
        nn_i = int(np.argmin(((Rf - fake[i].reshape(-1).astype(np.float32)) ** 2).sum(1)))
        ax[0, j].imshow(fake[i]); ax[1, j].imshow(real[nn_i]); ax[0, j].axis("off"); ax[1, j].axis("off")
    ax[0, 0].set_title("synthetic (closest 8)", loc="left", fontsize=8); ax[1, 0].set_title("nearest real", loc="left", fontsize=8)
    plt.tight_layout(); plt.savefig(OUT / f"figures/F07b_memorisation_client{k}.png", bbox_inches="tight"); plt.show()
    del G; torch.cuda.empty_cache()

T09 = pd.DataFrame(gan_rows); T09.to_csv(OUT / "tables/T09_gan_quality.csv", index=False); print(T09.round(3).to_string(index=False))

hist = json.load(open(OUT / "gan" / f"loss_a{ALPHA}_s{s0}_{PROFILE}.json"))
plt.figure(figsize=(7, 4))
for k, h in hist.items():
    ep = np.arange(1, len(h["g"]) + 1)
    plt.plot(ep, h["g"], color=f"C{k}", label=f"Client {k} — G"); plt.plot(ep, h["d"], "--", color=f"C{k}", label=f"Client {k} — D")
plt.xlabel("Epoch"); plt.ylabel("BCE loss"); plt.title("Client-side cGAN training"); plt.legend(ncol=3, fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "figures/F05_cgan_losses.png", bbox_inches="tight"); plt.show()

## 5 · Computational complexity

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

comp = []
for bb in [BACKBONE, "resnet50", "efficientnet_b0"]:
    m = new_model(bb).eval(); n_par = sum(p.numel() for p in m.parameters())
    x1 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    with FlopCounterMode(display=False) as fc:
        with torch.no_grad(): m(x1)
    flops = fc.get_total_flops()
    def _lat(bs, it=30):
        x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
        with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            for _ in range(5): m(x)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            t = time.time()
            for _ in range(it): m(x)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
        return (time.time() - t) / it
    l1, l32 = _lat(1), _lat(32)
    comp.append(dict(backbone=bb, params_M=n_par / 1e6, GFLOPs=flops / 1e9, model_MB_fp32=n_par * 4 / 1e6,
                     latency_ms_bs1=l1 * 1e3, throughput_img_s_bs32=32 / l32,
                     comm_MB_per_round_3clients=2 * NUM_CLIENTS * n_par * 4 / 1e6))
    del m; torch.cuda.empty_cache()
cg = ConditionalGenerator(); cd = ConditionalDiscriminator()
comp.append(dict(backbone="cGAN (G + D, client-only, never transmitted)",
                 params_M=(sum(p.numel() for p in cg.parameters()) + sum(p.numel() for p in cd.parameters())) / 1e6))
T10 = pd.DataFrame(comp); T10.to_csv(OUT / "tables/T10_complexity.csv", index=False); print(T10.round(2).to_string(index=False))

## 6 · Tables: main result, per-class, ablation, significance, robustness, sensitivity

In [ ]:
RESULTS = [json.load(open(f)) for f in sorted(glob.glob(str(RUNS_DIR / "*.json")))]
R = pd.DataFrame([dict(key=r["key"], name=r["name"], group=r["group"], seed=r["seed"], best_round=r.get("best_round"),
                       train_min=r["train_time_s"] / 60, comm_MB=r.get("comm_MB", 0.0),
                       **{m: r["test"][m] for m in SCALARS}) for r in RESULTS])
R.to_csv(OUT / "tables/all_runs_long.csv", index=False)
print(f"{len(R)} completed runs")

def load_pred(key, seed, suffix=""):
    z = np.load(RUNS_DIR / f"{key}__s{seed}{suffix}.npz"); return z["y"], z["prob"]

# ---- T04 main result ----
P = R[R.key == "proposed"].sort_values("seed")
rows = []
for m in SCALARS:
    mm, sd, lo, hi, n = mean_sd_ci(P[m].values)
    rows.append(dict(metric=m, mean=mm, sd=sd, ci95_low=lo, ci95_high=hi, n_seeds=n))
T04 = pd.DataFrame(rows)
y0, p0 = load_pred("proposed", SEEDS_MAIN[0])
boot = bootstrap_ci(y0, p0, B=BOOTSTRAP_B)
T04["bootstrap95_seed%d" % SEEDS_MAIN[0]] = T04.metric.map(lambda m: f"[{boot[m][0]:.4f}, {boot[m][1]:.4f}]" if m in boot else "")
T04.to_csv(OUT / "tables/T04_main_results.csv", index=False); print(T04.round(4).to_string(index=False))

# ---- T05 per-class (mean ± SD over seeds) ----
pc = defaultdict(lambda: defaultdict(list))
for r in RESULTS:
    if r["key"] == "proposed":
        for c, d in r["test"]["per_class"].items():
            for k, v in d.items(): pc[c][k].append(v)
T05 = pd.DataFrame([dict(cls=c, **{k: fmt(*mean_sd_ci(v)[:2], pct=(k != "support"), d=2 if k != "support" else 0)
                                  for k, v in d.items()}) for c, d in pc.items()])
T05.to_csv(OUT / "tables/T05_per_class.csv", index=False); print(T05.to_string(index=False))

# ---- T06 ablation / baselines ----
order = [e["key"] for e in EXPERIMENTS]
def summary_table(groups):
    out = []
    for key in order:
        d = R[(R.key == key) & (R.group.isin(groups))]
        if not len(d): continue
        row = dict(key=key, configuration=EXP_BY_KEY[key]["name"], n_seeds=len(d))
        for m in ["accuracy", "balanced_accuracy", "precision_macro", "recall_macro", "f1_macro", "auc_macro", "kappa", "mcc"]:
            mm, sd, *_ = mean_sd_ci(d[m].values); row[m] = fmt(mm, sd, pct=m not in ("kappa", "mcc"), d=2 if m not in ("kappa", "mcc") else 3)
        row["rounds_to_best"] = fmt(*mean_sd_ci(d.best_round.astype(float).values)[:2], pct=False, d=1)
        row["train_min"] = f"{d.train_min.mean():.1f}"; row["comm_MB"] = f"{d.comm_MB.mean():.0f}"
        out.append(row)
    return pd.DataFrame(out)
T06 = summary_table(["main", "ablation", "backbone"]); T06.to_csv(OUT / "tables/T06_ablation.csv", index=False)
print(T06.to_string(index=False))

# ---- T07 significance vs proposed (paired on shared seeds) ----
sig = []
for key in order:
    if key == "proposed" or EXP_BY_KEY[key]["group"] == "sensitivity": continue
    d = R[R.key == key]
    shared = sorted(set(d.seed) & set(P.seed))
    if not shared: continue
    a = P.set_index("seed").loc[shared, "f1_macro"].values; b = d.set_index("seed").loc[shared, "f1_macro"].values
    row = dict(key=key, configuration=EXP_BY_KEY[key]["name"], n_pairs=len(shared), delta_f1_macro_pp=100 * float(np.mean(a - b)))
    row["paired_t_p"] = float(stats.ttest_rel(a, b).pvalue) if len(shared) > 1 and np.std(a - b) > 0 else np.nan
    row["wilcoxon_p"] = float(stats.wilcoxon(a, b).pvalue) if len(shared) > 1 and np.any(a != b) else np.nan
    if EXP_BY_KEY[key]["mode"] != "local":
        ya, pa = load_pred("proposed", shared[0]); yb, pb = load_pred(key, shared[0])
        bb, cc, pm = mcnemar_exact(ya, pa.argmax(1), pb.argmax(1))
        row.update(mcnemar_seed=shared[0], proposed_only_correct=bb, other_only_correct=cc, mcnemar_p=pm)
    sig.append(row)
T07 = pd.DataFrame(sig)
if len(T07):
    for c in ["paired_t_p", "wilcoxon_p", "mcnemar_p"]:
        if c in T07: T07[c + "_holm"] = holm(T07[c].values)
fr_keys = [k for k in order if EXP_BY_KEY[k]["group"] in ("main", "ablation", "backbone")
           and set(SEEDS_OTHER) <= set(R[R.key == k].seed)]
common = set(SEEDS_OTHER)
friedman = None
if len(common) >= 2 and len(fr_keys) >= 3:
    mat = [R[(R.key == k) & (R.seed.isin(common))].sort_values("seed").f1_macro.values for k in fr_keys]
    friedman = stats.friedmanchisquare(*mat); print(f"Friedman (macro-F1, {len(fr_keys)} configs × {len(common)} seeds): "
                                                    f"χ²={friedman.statistic:.3f}, p={friedman.pvalue:.4g}")
T07.to_csv(OUT / "tables/T07_significance.csv", index=False); print(T07.round(4).to_string(index=False))
print("NOTE: with n paired seeds the smallest attainable two-sided Wilcoxon p is 2/2^n (n=3 -> 0.25, n=5 -> 0.0625); "
      "McNemar on the fixed test set is the primary per-sample test.")

# ---- T08 robustness ----
rob = []
for key in ["proposed", "partial"] + [f"loco_c{k}" for k in range(NUM_CLIENTS)]:
    rs = [r for r in RESULTS if r["key"] == key and (key == "proposed" and r["seed"] in SEEDS_OTHER or key != "proposed")]
    if not rs: continue
    row = dict(configuration=EXP_BY_KEY[key]["name"], n_seeds=len(rs))
    for m in ["accuracy", "f1_macro", "auc_macro"]:
        row["test_" + m] = fmt(*mean_sd_ci([r["test"][m] for r in rs])[:2])
    if "unseen_client" in rs[0]:
        for m in ["accuracy", "balanced_accuracy", "f1_macro", "auc_macro"]:
            row["unseen_client_" + m] = fmt(*mean_sd_ci([r["unseen_client"][m] for r in rs])[:2])
    rob.append(row)
T08 = pd.DataFrame(rob); T08.to_csv(OUT / "tables/T08_robustness.csv", index=False); print(T08.to_string(index=False))

# ---- T11 sensitivity ----
T11 = summary_table(["sensitivity"])
if len(T11): T11.to_csv(OUT / "tables/T11_sensitivity.csv", index=False); print(T11.to_string(index=False))

# ---- T03 hyper-parameters actually used ----
T03 = pd.DataFrame([
    ("Dataset", "Eye Diseases Classification (Kaggle), 4 classes"), ("Images used (after de-dup)", len(DF)),
    ("Split", f"patient/near-duplicate-grouped, stratified: train {len(TRAIN_IDX)} / val {len(VAL_IDX)} / test {len(TEST_IDX)}"),
    ("Preprocessing", "black-border crop, resize 224, illumination correction, NLM denoising, CLAHE (L channel), FOV mask"),
    ("Clients / partition", f"{NUM_CLIENTS} clients, patient-grouped Dirichlet α={ALPHA}, each ≥{int(MIN_CLIENT_FRAC * 100)}% of data"),
    ("Classifier", f"{BACKBONE} (timm, pretrained={PRETRAINED}), head dropout {DROP_RATE}"),
    ("Loss", f"class-balanced focal loss (β={CB_BETA}, γ={FOCAL_GAMMA}) with client-local class counts"),
    ("Optimiser", f"AdamW, lr {LR}, weight decay {WEIGHT_DECAY}, cosine decay over rounds, grad-clip {GRAD_CLIP}, AMP fp16"),
    ("Batch size", f"{BATCH} ({'DataParallel over %d GPUs' % N_GPU if N_GPU > 1 else 'single device'})" if 'N_GPU' in globals() else BATCH),
    ("Federated rounds / local epochs", f"max {ROUNDS} / {LOCAL_EPOCHS}; early stopping on val macro-AUC (patience {PATIENCE})"),
    ("Aggregation", f"FedAvg weighted by real-sample count; FedProx μ={MU}"),
    ("cGAN", f"{GAN_SIZE}×{GAN_SIZE}, latent {LATENT}, {GAN_EPOCHS} epochs, batch {GAN_BATCH}, Adam {GAN_LR} (β1={GAN_BETA1}), label smoothing 0.9"),
    ("Synthetic per class per client", f"min(gap to majority class, {SYNTH_CAP}); upsampled 64→224 bicubic; never re-preprocessed"),
    ("Seeds", f"proposed {SEEDS_MAIN}; others {SEEDS_OTHER}"),
], columns=["Setting", "Value"])
T03.to_csv(OUT / "tables/T03_hyperparameters.csv", index=False)

## 7 · Figures from logged outputs

In [ ]:
s0 = SEEDS_MAIN[0]
# --- convergence (mean ± SD over seeds) ---
H = pd.DataFrame([dict(seed=r["seed"], **h) for r in RESULTS if r["key"] == "proposed" for h in r["history"]])
if len(H):
    g = H.groupby("round"); fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    for a, cols, ttl in [(ax[0], [("train_loss", "train"), ("val_loss", "validation")], "Loss"),
                         (ax[1], [("train_acc", "train"), ("val_acc", "validation"), ("test_acc", "test")], "Accuracy"),
                         (ax[2], [("val_auc", "validation"), ("test_auc", "test")], "Macro-AUC")]:
        for c, lbl in cols:
            mu_, sd_ = g[c].mean(), g[c].std().fillna(0)
            a.plot(mu_.index, mu_.values, marker="o", ms=3, label=lbl); a.fill_between(mu_.index, mu_ - sd_, mu_ + sd_, alpha=0.2)
        a.set_xlabel("Federated round"); a.set_title(ttl); a.legend(); a.xaxis.get_major_locator().set_params(integer=True)
    plt.suptitle(f"FediGAN-ViT convergence (mean ± SD over {H.seed.nunique()} seeds)"); plt.tight_layout()
    plt.savefig(OUT / "figures/F08_convergence.png", bbox_inches="tight"); plt.show()
    H.to_csv(OUT / "tables/convergence_history.csv", index=False)

# --- confusion matrices ---
cms = [np.array(r["test"]["cm"]) for r in RESULTS if r["key"] == "proposed"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, M, ttl, fm in [(ax[0], np.array([r for r in RESULTS if r["key"] == "proposed" and r["seed"] == s0][0]["test"]["cm"]), f"Counts (seed {s0}, n={len(TEST_IDX)})", "d"),
                      (ax[1], sum(cms) / sum(cms).sum(1, keepdims=True), f"Row-normalised, pooled over {len(cms)} seeds", ".3f")]:
    im = a.imshow(M, cmap="Blues"); a.set_xticks(range(NUM_CLASSES)); a.set_yticks(range(NUM_CLASSES))
    a.set_xticklabels(SHORT); a.set_yticklabels(SHORT); a.set_xlabel("Predicted"); a.set_ylabel("True"); a.set_title(ttl)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            a.text(j, i, format(M[i, j], fm), ha="center", va="center", color="white" if M[i, j] > M.max() / 2 else "black")
    plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.savefig(OUT / "figures/F09_confusion_matrix.png", bbox_inches="tight"); plt.show()

# --- ROC and PR (seed s0) ---
y0, p0 = load_pred("proposed", s0); yb = np.eye(NUM_CLASSES)[y0]
fig, ax = plt.subplots(1, 2, figsize=(12, 5)); grid = np.linspace(0, 1, 501); tprs = []
for c in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(yb[:, c], p0[:, c]); tprs.append(np.interp(grid, fpr, tpr))
    ax[0].plot(fpr, tpr, label=f"{SHORT[c]} (AUC = {roc_auc_score(yb[:, c], p0[:, c]):.3f})")
    pr, rc, _ = precision_recall_curve(yb[:, c], p0[:, c])
    ax[1].plot(rc, pr, label=f"{SHORT[c]} (AP = {average_precision_score(yb[:, c], p0[:, c]):.3f})")
ax[0].plot(grid, np.mean(tprs, 0), "k-", lw=2, label=f"Macro-average (AUC = {np.mean([roc_auc_score(yb[:, c], p0[:, c]) for c in range(NUM_CLASSES)]):.3f})")
ax[0].plot([0, 1], [0, 1], ":", color="grey"); ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate"); ax[0].set_title("ROC (one-vs-rest)")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision–recall")
for a in ax: a.legend(loc="lower right" if a is ax[0] else "lower left", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "figures/F10_roc_pr.png", bbox_inches="tight"); plt.show()

# --- ablation bars ---
keys = [k for k in order if k in set(R.key) and EXP_BY_KEY[k]["group"] in ("main", "ablation", "backbone")]
if keys:
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(keys) + 1.5)); yy = np.arange(len(keys))
    for off, m, lbl in [(-0.2, "f1_macro", "Macro-F1"), (0.2, "auc_macro", "Macro-AUC")]:
        mm = [R[R.key == k][m].mean() * 100 for k in keys]; sd = [np.nan_to_num(R[R.key == k][m].std()) * 100 for k in keys]
        ax.barh(yy + off, mm, 0.38, xerr=sd, label=lbl, capsize=2)
    ax.set_yticks(yy); ax.set_yticklabels([EXP_BY_KEY[k]["name"] for k in keys], fontsize=8); ax.invert_yaxis()
    lo = min(R[R.key.isin(keys)][["f1_macro", "auc_macro"]].min()) * 100
    ax.set_xlim(max(0, lo - 5), 100); ax.set_xlabel("% (mean ± SD over seeds)"); ax.legend(); ax.set_title("Ablation and baselines")
    plt.tight_layout(); plt.savefig(OUT / "figures/F13_ablation.png", bbox_inches="tight"); plt.show()

# --- sample predictions ---
rng = np.random.default_rng(1); pick = rng.choice(len(TEST_IDX), 10, replace=False)
fig, ax = plt.subplots(2, 5, figsize=(14, 6))
for a, i in zip(ax.flat, pick):
    a.imshow(RAW[TEST_IDX[i]]); ok = p0[i].argmax() == y0[i]
    a.set_title(f"True: {SHORT[y0[i]]}\nPred: {SHORT[p0[i].argmax()]} ({p0[i].max():.2f})", color="green" if ok else "red", fontsize=9); a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "figures/F11_test_predictions.png", bbox_inches="tight"); plt.show()

## 8 · Grad-CAM on the proposed global model (distinct test images, correct and incorrect)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

model = new_model(BACKBONE); model.load_state_dict(torch.load(OUT / "models" / f"proposed_s{s0}_{RUN_TAG}.pth", map_location=DEVICE)); model.eval()
_, tf_test = make_tfs(_INIT[BACKBONE][1]["mean"], _INIT[BACKBONE][1]["std"])
if "vit" in BACKBONE:
    npt = model.num_prefix_tokens; gsz = IMG_SIZE // model.patch_embed.patch_size[0]
    reshape = lambda t: t[:, npt:, :].reshape(t.size(0), gsz, gsz, t.size(2)).permute(0, 3, 1, 2)
    cam = GradCAM(model=model, target_layers=[model.blocks[-1].norm1], reshape_transform=reshape)
else:
    cam = GradCAM(model=model, target_layers=[model.layer4[-1]] if hasattr(model, "layer4") else [model.conv_head])

pred0 = p0.argmax(1); sel = []
for c in range(NUM_CLASSES):
    ok = np.where((y0 == c) & (pred0 == c))[0]; bad = np.where((y0 == c) & (pred0 != c))[0]
    sel += list(ok[np.argsort(-p0[ok, c])][:2]) + list(bad[:1])
fig, ax = plt.subplots(len(sel), 4, figsize=(10, 2.6 * len(sel)))
for r, i in enumerate(sel):
    ai = TEST_IDX[i]
    x = tf_test(tv_tensors.Image(torch.from_numpy(PROC[ai]).permute(2, 0, 1)))[None].to(DEVICE)
    g = cam(input_tensor=x, targets=[ClassifierOutputTarget(int(pred0[i]))])[0]
    rawf = RAW[ai].astype(np.float32) / 255
    for j, (img, ttl) in enumerate([(RAW[ai], "Original"), (PROC[ai], "Model input"), (g, "Grad-CAM"),
                                     (show_cam_on_image(rawf, g, use_rgb=True), "Overlay")]):
        ax[r, j].imshow(img, cmap="jet" if j == 2 else None); ax[r, j].axis("off")
        if r == 0: ax[r, j].set_title(ttl)
    ax[r, 0].text(-10, IMG_SIZE / 2, f"T: {SHORT[y0[i]]}\nP: {SHORT[pred0[i]]} ({p0[i].max():.2f})", ha="right", va="center",
                  fontsize=8, color="green" if y0[i] == pred0[i] else "red")
plt.tight_layout(); plt.savefig(OUT / "figures/F12_gradcam.png", bbox_inches="tight"); plt.show()

## 9 · Results summary + downloadable archive

In [ ]:
def _row(t, m): r = t[t.metric == m].iloc[0]; return f"{100 * r['mean']:.2f} ± {100 * r['sd']:.2f}" if not np.isnan(r['sd']) else f"{100 * r['mean']:.2f}"
lines = [f"# FediGAN-ViT — results summary ({RUN_TAG}, backbone {BACKBONE})", "",
         f"Runs completed: {len(R)}; skipped for time: {sum(len(json.load(open(f))['skipped']) for f in glob.glob(str(OUT / f'skipped_{RUN_TAG}*.json')))}",
         f"Test set: {len(TEST_IDX)} real images (patient-grouped, never seen in training, no synthetic images).", "",
         f"## Proposed model (mean ± SD over {int(T04.n_seeds.max())} seeds)", ""]
for m in ["accuracy", "balanced_accuracy", "precision_macro", "recall_macro", "f1_macro", "auc_macro"]:
    lines.append(f"- {m}: {_row(T04, m)} %")
lines += [f"- Cohen's κ: {T04[T04.metric == 'kappa']['mean'].iloc[0]:.3f}; MCC: {T04[T04.metric == 'mcc']['mean'].iloc[0]:.3f}", "",
          "## Tables", *[f"- `{p.name}`" for p in sorted((OUT / 'tables').glob('*.csv'))], "",
          "## Figures", *[f"- `{p.name}`" for p in sorted((OUT / 'figures').glob('*.png'))], "",
          "## Wording to use in the paper",
          "- Report macro-averaged metrics (classes are balanced overall but not per client).",
          "- Inception Score is computed on ImageNet classes and is weakly meaningful for fundus images; FID/KID are reported with a real-vs-real floor.",
          "- Validation for model selection is a held-out server set (simulation); state this explicitly.",
          "- No formal privacy mechanism (DP / secure aggregation) is used; describe the framework as 'data-local', not 'privacy-guaranteed'."]
Path(OUT / "RESULTS_SUMMARY.md").write_text("\n".join(lines)); print("\n".join(lines))

import zipfile
zpath = WORK / f"fedigan_results_{RUN_TAG}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        rel = f.relative_to(OUT)
        if f.is_file() and rel.parts[0] not in ("cache", "models", "synthetic", "gan") and \
                (not rel.parts[0].startswith("runs_") or rel.parts[0] == f"runs_{RUN_TAG}"):
            z.write(f, rel)
print("Download:", zpath, f"({zpath.stat().st_size / 1e6:.1f} MB)")